# Tutorial 8: Exam Preparation — Regression, Validation, and PCA

**Big Data in Finance** | Imperial College Business School  
**Duration:** 1 hour

---

## Purpose

This tutorial prepares you for **Section B (Applied Problems)** of the final exam. The exercises below are structured like exam questions: you are given code snippets, model outputs, or scenarios and asked to **interpret**, **debug**, or **calculate**.

### How to Use This Tutorial

1. **Try each exercise on paper first** — this is how you'll work in the exam
2. Then use the code cells to **verify your answers**
3. Solutions are provided at the end of each exercise — try before you peek!

### Coverage

- Exercise 1: Debugging a Ridge regression pipeline (data leakage)
- Exercise 2: Interpreting OOS R² and Lasso coefficient paths
- Exercise 3: PCA — variance explained and proper implementation
- Exercise 4: Comparing models from walk-forward validation output

---

## Setup

Run the cell below to load libraries. You'll use these to verify your answers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import TimeSeriesSplit, KFold

np.random.seed(42)

---

## Exercise 1: Debugging a Lasso Pipeline [~15 minutes]

A colleague has written the following code to predict monthly market returns using Lasso regression with macroeconomic predictors. The code runs without errors, but the colleague is suspicious because the out-of-sample R² is unusually high (4.8%).

**The code contains THREE mistakes** that could lead to inflated performance. Identify each mistake, explain why it is problematic, and provide the corrected code.

```python
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold

# Load data (sorted chronologically)
data = pd.read_csv('macro_predictors.csv')
X = data.drop(columns=['date', 'market_return'])
y = data['market_return']

# Step 1: Reduce dimensionality with PCA
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X)                          # Line A

# Step 2: Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca)                # Line B

# Step 3: Time-series split
n = len(X_scaled)
split = int(0.8 * n)
X_train, X_test = X_scaled[:split], X_scaled[split:]
y_train, y_test = y[:split], y[split:]

# Step 4: Fit Lasso with cross-validation
model = LassoCV(alphas=np.logspace(-4, 1, 50),
                cv=KFold(n_splits=5, shuffle=True),    # Line C
                max_iter=10000)
model.fit(X_train, y_train)

# Step 5: Evaluate
y_pred = model.predict(X_test)
oos_r2 = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - y_test.mean())**2)
print(f'OOS R²: {oos_r2:.4f}')
```

### ✍️ Your Answer

Write your answer here before looking at the solution:

**Mistake 1 (Line __):**
- Problem: ...
- Fix: ...

**Mistake 2 (Line __):**
- Problem: ...
- Fix: ...

**Mistake 3 (Line __):**
- Problem: ...
- Fix: ...

### 💡 Solution

<details>
<summary>Click to reveal</summary>

**Mistake 1 — Line A: PCA fitted on full data before splitting**

- **Problem:** `pca.fit_transform(X)` computes the principal component directions using *all* observations, including the test set. This means the PCA transformation incorporates future information, creating data leakage. The test set is no longer truly out-of-sample because the feature transformation already "knows" about its structure.
- **Fix:** Split the data first, then fit PCA only on the training set:
```python
pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train_raw)
X_test_pca = pca.transform(X_test_raw)  # transform only, do NOT fit
```

**Mistake 2 — Line B: Scaling fitted on full data before splitting**

- **Problem:** Same issue as PCA — `scaler.fit_transform(X_pca)` computes the mean and standard deviation using all observations (including test data). The training features are influenced by test-set statistics.
- **Fix:** After splitting, fit the scaler on training data only:
```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled = scaler.transform(X_test_pca)
```

**Note:** Lines A and B share the same root cause — preprocessing on the full dataset before splitting. The correct order is always: (1) split, (2) fit preprocessing on training, (3) transform both.

**Mistake 3 — Line C: KFold with shuffle inside LassoCV**

- **Problem:** `KFold(n_splits=5, shuffle=True)` randomly shuffles the training data before creating folds. This means that within the hyperparameter tuning step, the model may train on observations from 2015 and validate on observations from 2010, creating look-ahead bias *within the training set*.
- **Fix:** Use `TimeSeriesSplit`:
```python
from sklearn.model_selection import TimeSeriesSplit
model = LassoCV(alphas=np.logspace(-4, 1, 50),
                cv=TimeSeriesSplit(n_splits=5),
                max_iter=10000)
```

**Bonus mistake (not counted but worth noting):** The OOS R² denominator uses `y_test.mean()` instead of `y_train.mean()`. The proper benchmark is the training set mean, which is what you would use in real-time forecasting.

</details>

---

## Exercise 2: Interpreting Ridge and Lasso Output [~15 minutes]

You fit Ridge and Lasso regression to predict quarterly market returns using 10 macroeconomic predictors. Below are the results.

### Part (a): Interpreting Coefficient Paths

The following table shows Lasso coefficients for three values of λ:

| Predictor | λ = 0.001 | λ = 0.01 | λ = 0.1 |
|-----------|-----------|----------|----------|
| d_p | -0.032 | -0.028 | 0.000 |
| e_p | 0.045 | 0.041 | 0.025 |
| b_m | 0.018 | 0.012 | 0.000 |
| tbl | -0.021 | -0.015 | 0.000 |
| tms | 0.039 | 0.035 | 0.019 |
| dfy | 0.008 | 0.000 | 0.000 |
| infl | -0.005 | 0.000 | 0.000 |
| svar | -0.027 | -0.022 | -0.011 |
| ntis | 0.003 | 0.000 | 0.000 |
| ltr | 0.011 | 0.006 | 0.000 |

**Questions:**

1. Which three predictors does Lasso consider most important? How can you tell?
2. If time-series CV selects λ = 0.01, how many predictors are retained in the model?
3. Your colleague argues that since Ridge never sets coefficients to zero, it is always worse than Lasso for prediction. Is this correct? Explain.

### ✍️ Your Answer

1. ...
2. ...
3. ...

### Part (b): Comparing OOS Performance

You evaluate both models out-of-sample using a walk-forward procedure. The results are:

| Metric | Ridge (λ=5.0) | Lasso (λ=0.01) | Historical Mean |
|--------|:---:|:---:|:---:|
| OOS R² | 0.8% | 1.2% | 0.0% (by definition) |
| Directional accuracy | 56.3% | 54.1% | 50.0% |
| Annualised Sharpe (long/short) | 0.42 | 0.51 | 0.38 |
| Annual turnover | 1.8x | 3.2x | 1.0x |

**Questions:**

4. Both models have OOS R² close to 1%. Is this meaningful or practically zero? Justify your answer.
5. Which model would you recommend to an investor? Consider both statistical and economic criteria.
6. A transaction cost of 10 basis points per unit of turnover would reduce the Sharpe ratio. Which model is more affected by this, and why?

### ✍️ Your Answer

4. ...
5. ...
6. ...

### 💡 Solution

<details>
<summary>Click to reveal</summary>

**1.** The three most important predictors are **e_p** (earnings-price ratio), **tms** (term spread), and **svar** (stock variance). These are identifiable because they are the *last* predictors to be set to zero as λ increases — they survive even at λ = 0.1 (e_p and tms with non-zero coefficients, svar as well). Lasso's feature selection property means the predictors that survive strong regularization carry the most robust signal.

**2.** At λ = 0.01, predictors with non-zero coefficients are: d_p, e_p, b_m, tbl, tms, svar, ltr — that is **7 out of 10** predictors retained. Three are eliminated: dfy, infl, ntis.

**3.** This is **not correct**. Ridge shrinks all coefficients but never eliminates them, while Lasso performs feature selection. However, when many predictors have small but non-zero true effects, Ridge can outperform Lasso because Lasso may wrongly eliminate some useful predictors. Ridge is particularly strong when predictors are correlated (common in macro data), because Lasso tends to arbitrarily select one from a group of correlated features and discard the rest. The best method depends on the data — there is no universal winner.

**4.** An OOS R² of ~1% is **economically meaningful** for equity premium prediction. The signal-to-noise ratio in market returns is very low: monthly return volatility is ~4-5% while the expected premium is ~0.5%. Most return variation is genuinely unpredictable. Campbell & Thompson (2008) show that even R² ≈ 0.5% can generate substantial utility gains for investors. The fact that both models beat the historical mean benchmark (R² > 0%) is a meaningful result.

**5.** This requires weighing several factors. Lasso has higher OOS R² (1.2% vs 0.8%) and a better Sharpe ratio (0.51 vs 0.42). However, Ridge has better directional accuracy (56.3% vs 54.1%) and much lower turnover (1.8x vs 3.2x). **After transaction costs**, the performance gap narrows or may reverse (see Q6). A practical recommendation might be Ridge for its lower turnover and higher directional accuracy, or Lasso if transaction costs are very low. A good answer discusses the tradeoff rather than picking one outright.

**6.** Lasso is more affected. With 10bp per unit of turnover: Lasso incurs 3.2 × 10bp = 32bp in annual costs, while Ridge incurs 1.8 × 10bp = 18bp. Lasso's higher turnover comes from its sparse coefficient structure — as the selected features change across rolling windows, the portfolio can shift dramatically. Ridge's smoother coefficient updates lead to more stable portfolio weights and lower trading costs. This illustrates a general principle: models that are more "aggressive" in feature selection can generate more portfolio turnover.

</details>

---

## Exercise 3: PCA — Interpretation and Implementation [~15 minutes]

You perform PCA on 15 macroeconomic predictors. The scree plot data is shown below.

| Component | Variance Explained (%) | Cumulative (%) |
|:---------:|:-----:|:---------:|
| PC1 | 28.4 | 28.4 |
| PC2 | 16.1 | 44.5 |
| PC3 | 11.8 | 56.3 |
| PC4 | 9.2 | 65.5 |
| PC5 | 7.5 | 73.0 |
| PC6 | 5.3 | 78.3 |
| PC7 | 4.1 | 82.4 |
| PC8 | 3.2 | 85.6 |

### Part (a)

1. Using the "80% cumulative variance" rule, how many components would you retain?
2. The first component explains 28.4% of variance. In the context of macroeconomic predictors, what might this component capture? (Think about what common factor drives most macro variables together.)
3. Your colleague suggests retaining only PC1 because it has the highest variance explained. Is this a good idea? Why or why not?

### Part (b): Code Completion

Complete the missing parts of this code (marked `???`) to correctly implement PCA in a time-series prediction pipeline:

```python
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Data is already split chronologically
# X_train: (400, 15), X_test: (100, 15)

# Step 1: Scale features
scaler = StandardScaler()
X_train_sc = scaler.???              # (i)
X_test_sc = scaler.???               # (ii)

# Step 2: Apply PCA
pca = PCA(n_components=???)
X_train_pca = pca.???                # (iii)
X_test_pca = pca.???                 # (iv)

# Step 3: Fit Ridge on PCA features
model = Ridge(alpha=1.0)
model.fit(???, ???)                  # (v)
y_pred = model.predict(???)          # (vi)

# Step 4: Evaluate
train_mean = ???                     # (vii)
oos_r2 = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - ???)**2)  # (viii)
```

### ✍️ Your Answer

**Part (a):**
1. ...
2. ...
3. ...

**Part (b):**
- (i): ...
- (ii): ...
- (iii): ...
- (iv): ...
- (v): ...
- (vi): ...
- (vii): ...
- (viii): ...

### 💡 Solution

<details>
<summary>Click to reveal</summary>

**Part (a):**

1. **7 components.** The cumulative variance at PC7 is 82.4%, which is the first point exceeding 80%. PC6 at 78.3% is just below the threshold.

2. PC1 likely captures the **overall state of the economy** — a broad factor that drives most macroeconomic variables in the same direction. When the economy is strong, variables like earnings, dividends, and output tend to move together. This is analogous to a "market factor" in equity returns. The fact that it explains 28.4% (much more than any other component) suggests one dominant common factor in the data.

3. **No, this is a bad idea.** While PC1 captures the most variance, the remaining components can contain information that is *different* from PC1 and valuable for prediction. PC2–PC7 together explain 82.4% - 28.4% = 54.0% of variance — more than PC1 alone. Moreover, the component that best explains the *variance of X* is not necessarily the component that best *predicts Y*. In financial applications, some of the smaller components may capture predictive signals (e.g., a credit conditions factor) that are orthogonal to the dominant macro trend.

**Part (b):**

```python
# Step 1: Scale features
X_train_sc = scaler.fit_transform(X_train)    # (i) fit AND transform on training
X_test_sc = scaler.transform(X_test)           # (ii) transform only — no fit!

# Step 2: Apply PCA
pca = PCA(n_components=7)                      # from 80% rule
X_train_pca = pca.fit_transform(X_train_sc)    # (iii) fit AND transform on training
X_test_pca = pca.transform(X_test_sc)          # (iv) transform only — no fit!

# Step 3: Fit and predict
model.fit(X_train_pca, y_train)                # (v) fit on training PCA features
y_pred = model.predict(X_test_pca)             # (vi) predict on test PCA features

# Step 4: Evaluate
train_mean = y_train.mean()                    # (vii) training set mean
oos_r2 = 1 - ... / np.sum((y_test - train_mean)**2)  # (viii) use train mean
```

The critical pattern: `fit_transform` on training data, `transform` on test data. This applies to **both** the scaler and PCA. Using the training mean as the OOS R² benchmark ensures we're comparing against what a naive forecaster (using only past data) would predict.

</details>

---

## Exercise 4: Walk-Forward Validation Output [~15 minutes]

You run a walk-forward validation comparing four regression methods for predicting monthly market excess returns. The expanding-window procedure uses an initial training window of 240 months (20 years). Results over the 180-month test period:

| Method | OOS R² (%) | Mean Return (ann.) | Volatility (ann.) | Sharpe Ratio | Max Drawdown |
|--------|:---:|:---:|:---:|:---:|:---:|
| Historical Mean | 0.00 | 6.2% | 15.8% | 0.39 | -52.1% |
| Ridge | -0.35 | 7.1% | 14.2% | 0.50 | -41.3% |
| Random Forest | 1.82 | 6.8% | 16.5% | 0.41 | -48.7% |
| XGBoost | 2.45 | 5.9% | 17.1% | 0.35 | -55.2% |

**Questions:**

1. Ridge has a **negative** OOS R². Does this mean it is useless? Look at the other metrics before answering.

2. XGBoost has the **highest** OOS R² but the **lowest** Sharpe ratio. How is this possible? What does this tell us about the relationship between statistical and economic performance?

3. If you were advising a fund manager, which model would you recommend and why? Consider risk-adjusted performance, not just returns.

4. The Random Forest has higher volatility (16.5%) than the Historical Mean benchmark (15.8%) despite using the same return series. What could cause this?

### ✍️ Your Answer

1. ...
2. ...
3. ...
4. ...

### 💡 Solution

<details>
<summary>Click to reveal</summary>

**1.** No, Ridge is **not useless** despite its negative OOS R². The negative R² means that Ridge's predictions are worse than the historical mean as a *point forecast* — the squared errors are larger. However, Ridge delivers the **best Sharpe ratio** (0.50) and the **lowest max drawdown** (-41.3%). This is a clear example of the distinction between statistical and economic performance. What matters for a trading strategy is the *direction* of the prediction and risk management, not the magnitude. Ridge's stable, shrinkage-based forecasts may produce better directional calls even when the magnitudes are off, and the lower volatility (14.2%) suggests it avoids extreme bets.

**2.** XGBoost has the best OOS R² (meaning its point forecasts are closest to the realized returns on average) but the worst Sharpe ratio. This can happen because:
- R² measures average forecast accuracy across all periods, but Sharpe ratio depends on the *timing* of correct and incorrect predictions
- XGBoost may make very accurate predictions during calm periods but wrong predictions during volatile periods (when errors are most costly)
- The higher volatility (17.1%) suggests XGBoost makes more aggressive bets, amplifying losses when predictions are wrong
- R² weights all periods equally, while Sharpe penalizes periods of high loss more through the volatility denominator

This illustrates a key course theme: **statistical accuracy ≠ economic value**.

**3.** **Ridge** would be the recommended model for a fund manager. It has the highest Sharpe ratio (0.50), lowest volatility (14.2%), and shallowest max drawdown (-41.3%). Fund managers care about *risk-adjusted* returns, not just raw prediction accuracy. The lower drawdown also matters for client retention — investors tend to withdraw capital after large losses. Ridge's simpler, more stable predictions translate into more consistent investment performance, which is what practitioners value most.

**4.** The Historical Mean benchmark is a **buy-and-hold** strategy (always invest in the market with the same weight). A model-based strategy can have *higher* volatility if the model sometimes predicts large positive returns (leading to larger positions) or sometimes predicts negative returns (leading to short positions). The Random Forest's time-varying predictions cause the portfolio exposure to fluctuate, which can amplify volatility relative to a static buy-and-hold allocation. This is the cost of trying to time the market — even if the predictions add value on average, the varying exposure increases portfolio variability.

</details>

---

## 🎯 Key Takeaways for the Exam

1. **Data leakage** comes in many forms: scaling, PCA, feature engineering — anything fitted on full data before splitting
2. **The correct order is always:** split → fit on train → transform both train and test
3. **OOS R² can be negative** — and the model can still be economically useful
4. **Statistical vs. economic metrics** may tell different stories — always report both
5. **Transaction costs matter:** higher turnover erodes Sharpe ratios
6. **Lasso ≠ always better than Ridge:** when predictors are correlated, Ridge can outperform
7. **PCA:** fit only on training data, use the 80% variance rule as a starting point, and remember that the highest-variance components are not necessarily the most predictive